In [1]:
import pandas as pd
data_train = pd.read_parquet(r"final_data\chunk-based-split-3\train_df_prepared.parquet")
data_valid = pd.read_parquet(r"final_data\chunk-based-split-3\valid_df_prepared.parquet")
data_test = pd.read_parquet(r"final_data\chunk-based-split-3\test_df_prepared.parquet")

In [ ]:
import pandas as pd
import numpy as np
import torch
import gc
from torch_geometric.data import Data # thư viên Data giúp lưu trữ đồ thị lưới 
from tqdm.notebook import tqdm
import os

# bật cảnh báo trùng lặp của KMP (Intel MKL) để tránh lỗi crash khi dùng nhiều luồng trên CPU
os.environ["KMP_DUPLICATE_BLOCK_WARNING"] = "TRUE"

# Xây dựng đồ thị Flow-based Graph có đặc trưng cạnh
def build_ip_topology_graph(df, train_mask, valid_mask, test_mask, window_size=10):
    print(f"Bước 1: Trích xuất Node Features (X) và Label (y)...")
    
    cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port', 'label']
    
    # các cột cần drop 
    cols_present = [c for c in cols_to_drop if c in df.columns]

    # các cột còn lại sẽ là feature
    feature_cols = [c for c in df.columns if c not in cols_present]
    
    # tạo một numpy array chứa các feature
    features_np = np.empty((len(df), len(feature_cols)), dtype=np.float32)
    for i, col in enumerate(feature_cols):
        features_np[:, i] = df[col].astype(np.float32).values
    
    # ascontiguousarray: đảm bảo dữ liệu được lưu trữ liên tục trong bộ nhớ
    features_np = np.ascontiguousarray(features_np)
    
    # lấy node features (x_tensor) và label (y_tensor)
    x_tensor = torch.from_numpy(features_np).contiguous()
    y_tensor = torch.tensor(df['label'].values, dtype=torch.long).contiguous()



    print(f"Bước 2: Xây Dựng Đồ Thị Dây Chuyền Theo IP (Temporal IP Topology)...")

    # lấy các giá trị để xây dựng cạnh
    timestamps = df['timestamp_num'].values
    src_ips = df['src_ip'].values
    dst_ips = df['dst_ip'].values
    src_ports = df['src_port'].astype(str).values
    dst_ports = df['dst_port'].astype(str).values
    
    # tạo một mảng split_id để đánh dấu tập Train (0), Valid (1), Test (2)
    split_id = np.zeros(len(df), dtype=np.int8)
    split_id[valid_mask.numpy()] = 1
    split_id[test_mask.numpy()] = 2
    

    # gom nhóm các gói tin theo ip để nối cạnh
    ip_to_indices = {}
    for idx in tqdm(range(len(df)), desc="[1/2] Gom nhóm Gói tin theo IP"):
        s_ip = src_ips[idx]
        d_ip = dst_ips[idx]
        
        # ip_to_indices: dictionary mapping mỗi IP (cả src và dst) đến danh sách các chỉ số của gói tin liên quan đến IP đó
        if s_ip not in ip_to_indices: ip_to_indices[s_ip] = []
        if len(ip_to_indices[s_ip]) == 0 or ip_to_indices[s_ip][-1] != idx:
            ip_to_indices[s_ip].append(idx)
            
        if d_ip not in ip_to_indices: ip_to_indices[d_ip] = []
        if len(ip_to_indices[d_ip]) == 0 or ip_to_indices[d_ip][-1] != idx:
            ip_to_indices[d_ip].append(idx)
            
    all_src, all_dst, all_edge_attrs = [], [], []
    
    # nối cạnh giữa các gói tin có cùng IP (src hoặc dst) trong một cửa sổ thời gian nhất định (window_size)
    for ip, indices in tqdm(ip_to_indices.items(), desc=f"[2/2] Căng lưới đồ thị (Window = {window_size})"):
        n_flows = len(indices)
        if n_flows < 2: continue
        
        for i in range(1, n_flows):
            curr_idx = indices[i]
            start_w = max(0, i - window_size) # start_w: chỉ số bắt đầu của cửa sổ (window) để tìm các gói tin trước đó có cùng IP
            
            # duyệt qua các gói tin trước đó trong cửa sổ để tạo cạnh
            for j in range(start_w, i):
                past_idx = indices[j]
                
                dt_raw = abs(timestamps[curr_idx] - timestamps[past_idx]) # tính khoảng thời gian giữa 2 gói tin
                
                # tuy nhiên, nếu 2 gói tin cách nhau hơn 60s thì không nói cạnh 
                if dt_raw > 60.0:
                    continue
                
                # tính toán các đặc trưng cạnh
                # chênh lệch thời gian: (Log scale + Min-Max Scaling về dải [0, 1])
                # giá trị max lý thuyết khi dt_raw = 60s là log1p(60,000,000) ~ 17.9 => chia cho 18
                dt = np.log1p(dt_raw * 1e6) / 18.0
                # binary liệu 2 gói tin này có chung địa chỉ IP nguồn không (cùng người gửi với nhau)
                same_dir = 1.0 if src_ips[curr_idx] == src_ips[past_idx] else 0.0
                # liệu 2 gói tin này có cùng cổng nguồn không?
                same_sport = 1.0 if src_ports[curr_idx] == src_ports[past_idx] else 0.0
                # Liệu 2 gói tin này có cùng cổng đích không?
                same_dport = 1.0 if dst_ports[curr_idx] == dst_ports[past_idx] else 0.0

                # list lưu các đặc trưng cạnh
                attr = [dt, same_dir, same_sport, same_dport]
                
                # Nối cạnh giữa 2 gói tin có cùng IP
                # quy tắc nối cạnh:
                # nếu 2 gói tin cùng trong tập valid hoặc tập test hoặc 1 gói 1 tập : nối 1 chiều từ gói tin trước đến gói tin sau
                # nếu 2 gói tin cùng trong tập train: nối 2 chiều giữa 2 gói tin
                
                # nếu 2 gói đều thuộc tập train
                if split_id[curr_idx] == 0 and split_id[past_idx] == 0:
                    # nối 2 chiều giữa curr_idx và past_idx
                    all_src.extend([curr_idx, past_idx])
                    all_dst.extend([past_idx, curr_idx])
                    all_edge_attrs.extend([attr, attr]) 
                else:
                    # nối 1 chiều theo thời gain đối với các gói tin thuộc tập valid/test
                    all_src.append(past_idx)
                    all_dst.append(curr_idx)
                    all_edge_attrs.append(attr) 
                    
    print("\nHợp nhất và tạo Cạnh (Culling Edge Index)...")
    # loại bỏ các cạnh trùng lặp (nếu có) và tạo edge_index, edge_attr
    
    # tạo dataframe tạm thời để bỏ cạnh trùng lặp
    df_edges = pd.DataFrame({
        'src': all_src, 
        'dst': all_dst,
        'attr0': [a[0] for a in all_edge_attrs],
        'attr1': [a[1] for a in all_edge_attrs],
        'attr2': [a[2] for a in all_edge_attrs],
        'attr3': [a[3] for a in all_edge_attrs]
    })
    
    # loại bỏ các cạnh trùng lặp (nếu có) dựa trên cột 'src' và 'dst'
    df_edges = df_edges.drop_duplicates(subset=['src', 'dst']).reset_index(drop=True)
    
    # edge_index: tensor 2 x num_edges, edge_attr: tensor num_edges x num_edge_features
    edge_index = torch.tensor(df_edges[['src', 'dst']].values.T, dtype=torch.long).contiguous()
    edge_attr = torch.tensor(df_edges[['attr0', 'attr1', 'attr2', 'attr3']].values, dtype=torch.float32).contiguous()
    
    print(f"HOÀN THIỆN ĐỒ THỊ TEMPORAL IP (CÓ EDGE FEATURES)!")
    print(f"   - Số Nodes : {x_tensor.size(0):,}")
    print(f"   - Số Edges : {edge_index.size(1):,}")
    print(f"   - Số Features/Cạnh : {edge_attr.shape[1]}")
    
    del features_np, df_edges, all_src, all_dst, all_edge_attrs
    gc.collect()
    
    # tạo đồ thị PyG Data với node features, edge index, edge attributes và label
    graph = Data(x=x_tensor, edge_index=edge_index, edge_attr=edge_attr, y=y_tensor)
    
    # gắn mask cho tập train, valid, test vào đồ thị
    graph.train_mask = train_mask
    graph.valid_mask = valid_mask
    graph.test_mask = test_mask
    return graph

In [3]:
import gc
from sklearn.preprocessing import LabelEncoder, QuantileTransformer
from torch_geometric.loader import NeighborLoader
import torch
import numpy as np
import pandas as pd

print("=== BƯỚC 1: TIỀN XỬ LÝ VÀ CHUẨN BỊ STRICT INDUCTIVE IP GRAPH ===")

# tạo 1 cột để đánh dấu tập dữ liệu (train/valid/test) trước khi gộp để tránh Data Leakage
print("Đánh dấu tập dữ liệu và gộp 3 DataFrames thành một Siêu Dataframe...")
data_train['split_tag'] = 0
data_valid['split_tag'] = 1
data_test['split_tag'] = 2

# concat lại thành một dataframe chuẩn
df_all = pd.concat([data_train, data_valid, data_test], ignore_index=True)

# GIẢI PHÓNG RAM NGAY LẬP TỨC: Xóa 3 mảng con sau khi đã có siêu mảng df_all
del data_train, data_valid, data_test
gc.collect()

# chuyển datetime sang số để sắp xếp và giữ lại micro giây
df_all['timestamp_num'] = pd.to_datetime(df_all['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

print("Sắp xếp toàn bộ gói tin lồng nhau theo trình tự thời gian (Chronological Sort)...")
# Sắp xếp tĩnh trực tiếp (INPLACE) để chặn Pandas tạo bản copy 9GB trong bộ nhớ RAM
df_all.sort_values(by='timestamp_num', inplace=True)
df_all.reset_index(drop=True, inplace=True)
gc.collect()

total_nodes = len(df_all)

# tạo mask cho đồ thị
print("Tái tạo Cấu trúc Masks động (Dynamic Masking) bảo tồn tính chặt chẽ sau khi Sort...")
train_mask = torch.tensor((df_all['split_tag'] == 0).values, dtype=torch.bool)
valid_mask = torch.tensor((df_all['split_tag'] == 1).values, dtype=torch.bool)
test_mask  = torch.tensor((df_all['split_tag'] == 2).values, dtype=torch.bool)

# xóa cột split_tag
df_all.drop(columns=['split_tag'], inplace=True)
gc.collect()

# in ra thông tin thống kê về số lượng node và phân bố mask
print(f" - Tổng Node: {total_nodes:,}")
print(f" - Train Mask : {train_mask.sum().item():,} ({train_mask.float().mean()*100:.1f}%)")
print(f" - Valid Mask : {valid_mask.sum().item():,} ({valid_mask.float().mean()*100:.1f}%)")
print(f" - Test Mask  : {test_mask.sum().item():,} ({test_mask.float().mean()*100:.1f}%)")

print("Hoàn tất Data Prep cho Masks và Sorting Thời gian.")

# build đồ thị với đặc trưng cạnh và gắn mask
print("\n=== BẮT ĐẦU QUÁ TRÌNH BUILD IP TOPOLOGY INDEX ===")
super_graph_faiss = build_ip_topology_graph(df_all, train_mask, valid_mask, test_mask, window_size=10)
print("\nĐồ thị tổng IP Topo Đã Gắn Mask Không Rò Rỉ Thành Công!")

# tạo neighborloader (tương tự như dataloaders)
print("\nSet up IP Graph NeighborLoaders...")
train_loader_faiss = NeighborLoader(super_graph_faiss, num_neighbors=[10, 5], input_nodes=super_graph_faiss.train_mask, batch_size=2048, shuffle=True, num_workers=0)
valid_loader_faiss = NeighborLoader(super_graph_faiss, num_neighbors=[10, 5], input_nodes=super_graph_faiss.valid_mask, batch_size=2048, shuffle=False, num_workers=0)
test_loader_faiss  = NeighborLoader(super_graph_faiss, num_neighbors=[10, 5], input_nodes=super_graph_faiss.test_mask,  batch_size=2048, shuffle=False, num_workers=0)

del df_all 
gc.collect()
print("Hoàn tất Data Prep cho IP Graph")

=== BƯỚC 1: TIỀN XỬ LÝ VÀ CHUẨN BỊ STRICT INDUCTIVE IP GRAPH ===
Đánh dấu tập dữ liệu và gộp 3 DataFrames thành một Siêu Dataframe...
Sắp xếp toàn bộ gói tin lồng nhau theo trình tự thời gian (Chronological Sort)...
Tái tạo Cấu trúc Masks động (Dynamic Masking) bảo tồn tính chặt chẽ sau khi Sort...
 - Tổng Node: 3,801,012
 - Train Mask : 2,470,638 (65.0%)
 - Valid Mask : 570,134 (15.0%)
 - Test Mask  : 760,240 (20.0%)
Hoàn tất Data Prep cho Masks và Sorting Thời gian.

=== BẮT ĐẦU QUÁ TRÌNH BUILD IP TOPOLOGY INDEX ===
Bước 1: Trích xuất Node Features (X) và Label (y)...
Bước 2: Xây Dựng Đồ Thị Dây Chuyền Theo IP (Temporal IP Topology)...


[1/2] Gom nhóm Gói tin theo IP:   0%|          | 0/3801012 [00:00<?, ?it/s]

[2/2] Căng lưới đồ thị (Window = 10):   0%|          | 0/23 [00:00<?, ?it/s]


Hợp nhất và tạo Cạnh (Culling Edge Index)...
HOÀN THIỆN ĐỒ THỊ TEMPORAL IP (CÓ EDGE FEATURES)!
   - Số Nodes : 3,801,012
   - Số Edges : 107,556,948
   - Số Features/Cạnh : 4

Đồ thị tổng IP Topo Đã Gắn Mask Không Rò Rỉ Thành Công!

Set up IP Graph NeighborLoaders...
Hoàn tất Data Prep cho IP Graph


In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import GATv2Conv
from torch_geometric.utils import dropout_edge
import torch.nn as nn
from tqdm.notebook import tqdm
import copy
import gc

# định nghĩa gat với 2 lớp
class GAT_Embedder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, embedding_dim, num_classes, heads=4, edge_dropout=0.1, edge_dim=4):
        super(GAT_Embedder, self).__init__()
        self.edge_dropout = edge_dropout 
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=heads, dropout=0.1, edge_dim=edge_dim)
        self.bn1 = nn.BatchNorm1d(hidden_channels * heads)
        self.conv2 = GATv2Conv(hidden_channels * heads, embedding_dim, heads=1, concat=False, dropout=0.1, edge_dim=edge_dim)
        self.bn2 = nn.BatchNorm1d(embedding_dim)
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, x, edge_index, edge_attr):
        edge_index_dp, edge_mask = dropout_edge(edge_index, p=self.edge_dropout, force_undirected=False, training=self.training)
        edge_attr_dp = edge_attr[edge_mask] if edge_attr is not None else None
        
        x = F.dropout(x, p=0.25, training=self.training)
        x = self.conv1(x, edge_index_dp, edge_attr=edge_attr_dp)
        x = self.bn1(x)
        x = F.elu(x)
        
        edge_index_dp2, edge_mask2 = dropout_edge(edge_index, p=self.edge_dropout, force_undirected=False, training=self.training)
        edge_attr_dp2 = edge_attr[edge_mask2] if edge_attr is not None else None
        
        x = F.dropout(x, p=0.25, training=self.training)
        embedding = self.conv2(x, edge_index_dp2, edge_attr=edge_attr_dp2)
        embedding = self.bn2(embedding)
        embedding = F.elu(embedding) 
        
        out = self.classifier(embedding)
        return out, embedding

def train_gat(model, train_loader, valid_loader, device, epochs=5, lr=0.005, class_weights=None, patience=7):
    from sklearn.metrics import f1_score
    import numpy as np

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-3) 
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=2, verbose=True
    )
    
    if class_weights is not None:
        criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
    else:
        criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
        
    best_val_f1 = -1.0
    best_model_weights = None
    no_improve_epochs = 0
    
    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        total_train_nodes = 0
        all_train_preds = []
        all_train_labels = []
        
        pbar_train = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        for batch in pbar_train:
            batch = batch.to(device)
            optimizer.zero_grad()
            
            out, emb = model(batch.x, batch.edge_index, batch.edge_attr)
            batch_size = batch.batch_size
            
            loss = criterion(out[:batch_size], batch.y[:batch_size])
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            
            optimizer.step()
            
            pred = out[:batch_size].argmax(dim=1)
            y_true = batch.y[:batch_size]
            
            all_train_preds.append(pred.cpu().numpy())
            all_train_labels.append(y_true.cpu().numpy())
            
            total_train_loss += loss.item() * batch_size
            total_train_nodes += batch_size
            
            pbar_train.set_postfix({'Loss': f"{loss.item():.4f}"})
            
            del batch, out, emb, loss, pred, y_true
            
        avg_train_loss = total_train_loss / total_train_nodes
        train_f1 = f1_score(np.concatenate(all_train_labels), np.concatenate(all_train_preds), average='macro')
        
        pbar_train.close()
            
        model.eval()
        total_val_loss = 0
        total_val_nodes = 0
        all_val_preds = []
        all_val_labels = []
        
        with torch.no_grad():
            pbar_val = tqdm(valid_loader, desc=f"Epoch {epoch+1}/{epochs} [Valid]")
            for batch in pbar_val:
                batch = batch.to(device)
                
                out, emb = model(batch.x, batch.edge_index, batch.edge_attr)
                batch_size = batch.batch_size
                
                loss = criterion(out[:batch_size], batch.y[:batch_size])
                pred = out[:batch_size].argmax(dim=1)
                y_true = batch.y[:batch_size]
                
                all_val_preds.append(pred.cpu().numpy())
                all_val_labels.append(y_true.cpu().numpy())
                
                total_val_loss += loss.item() * batch_size
                total_val_nodes += batch_size
                
                pbar_val.set_postfix({'Loss': f"{loss.item():.4f}"})
                
                del batch, out, emb, loss, pred, y_true
                
        pbar_val.close()
                
        avg_val_loss = total_val_loss / total_val_nodes
        val_f1 = f1_score(np.concatenate(all_val_labels), np.concatenate(all_val_preds), average='macro')
        
        # SỬA: Theo dõi độ cải thiện của MACRO F1
        scheduler.step(val_f1)
        
        print(f"=== Epoch {epoch+1} Tổng kết ===")
        print(f"   Train Loss: {avg_train_loss:.4f} | Train Macro F1: {train_f1:.4f}")
        print(f"   Valid Loss: {avg_val_loss:.4f}   | Valid Macro F1: {val_f1:.4f}")
        current_lr = optimizer.param_groups[0]['lr']
        print(f"   Learning Rate hiện tại: {current_lr:.6f}")
        
        # SỬA: So sánh F1 hiện tại với F1 tốt nhất
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            no_improve_epochs = 0
            best_model_weights = copy.deepcopy(model.state_dict())
            print(f"Best model. Đã lưu trọng số tốt nhất tại Epoch {epoch+1} (Val Macro F1: {best_val_f1:.4f} | Val Loss: {avg_val_loss:.4f})")
        else:
            no_improve_epochs += 1
            print(f"Báo động Early Stopping: Không cải thiện F1 {no_improve_epochs}/{patience} epochs.")
            if no_improve_epochs >= patience:
                print(f"\nĐã kích hoạt Early Stopping tại Epoch {epoch+1}! Lùi về checkpoint tốt nhất.")
                break
                
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if best_model_weights is not None:
        model.load_state_dict(best_model_weights)
        print(f"\nHoàn tất Train! Đã load lại weights MANG LẠI F1 CAO NHẤT (Val Macro F1: {best_val_f1:.4f}) để dùng trích xuất Embedding!")
        
    return model

In [ ]:
import xgboost as xgb
from sklearn.metrics import classification_report, accuracy_score, f1_score
from tqdm.notebook import tqdm
import numpy as np
import torch
import gc
from torch_geometric.loader import NeighborLoader

# hàm trích xuất embedding từ model GAT
@torch.no_grad()
def extract_embeddings_mask(model, graph_data, mask, device='cpu'):
    model.eval()
    print("Đang khởi tạo Inference Loader múc lấy Embeddings...")
    
    # loader dành cho lúc test/extract
    loader = NeighborLoader(
        graph_data,
        num_neighbors=[10, 5], 
        input_nodes=mask,
        batch_size=512,
        shuffle=False,        
        num_workers=0 
    )
    
    all_embeddings = []
    all_labels = []
    
    pbar = tqdm(loader, desc="Đang trích xuất Vectơ Embedding")
    for batch in pbar:
        batch = batch.to(device)
        
        # truyền qua model để lấy embedding
        _, emb = model(batch.x, batch.edge_index, batch.edge_attr)
        
        # lấy embedding của các node trong batch (chỉ lấy phần đầu tương ứng với batch_size vì NeighborLoader sẽ trả về cả node gốc và neighbor)
        bs = batch.batch_size
        all_embeddings.append(emb[:bs].cpu().numpy())
        all_labels.append(batch.y[:bs].cpu().numpy())
        
        # xóa đi batch, emb để tránh nghẽn RAM
        del batch, emb, _
        
    pbar.close()
    
    # dọn rác cho các biến không dùng nữa
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    # hợp final_embeddings thành ma trận (batch_size x embedding_dim) và final_labels thành vector (batch_size,)
    final_embeddings = np.concatenate(all_embeddings, axis=0)
    final_labels = np.concatenate(all_labels, axis=0)
    
    print(f"Trích xuất thành công! Kích thước Ma trận Embeddings: {final_embeddings.shape}")
    return final_embeddings, final_labels

# hàm huấn luyện và đánh giá XGBoost từ embedding của GAT
def train_evaluate_xgboost(X_train_emb, y_train, X_valid_emb, y_valid, X_test_emb, y_test):
    print("\n--- BẮT ĐẦU TRAIN XGBOOST TỪ BASE GAT EMBEDDING ---")

    # tính toán class weights cho XGBoost dựa trên phân bố lớp tập train
    from sklearn.utils.class_weight import compute_class_weight
    print("Đang tính toán Custom Smoothed Class Weights...")
    
    unique_classes = np.unique(y_train)
    raw_class_weights = compute_class_weight('balanced', classes=unique_classes, y=y_train)
    weights_dict = {c: np.sqrt(w) for c, w in zip(unique_classes, raw_class_weights)}
    
    # tinh chỉnh weight thủ công
    if 4 in weights_dict: weights_dict[4] *= 0.6
    if 6 in weights_dict: weights_dict[6] *= 1.5
    if 9 in weights_dict: weights_dict[9] *= 2.5
    if 1 in weights_dict: weights_dict[1] *= 0.8

    # chuyển đổi thành mảng Sample Weight cho tập train
    final_sample_weights = np.array([weights_dict[y] for y in y_train])

    # train model xgboost
    xgb_model = xgb.XGBClassifier(
        n_estimators=1000, # 1000 cây
        learning_rate=0.08,
        max_depth=5, # mỗi cây có độ sâu tối đa là 7
        min_child_weight=2,
        gamma=1.0,
        reg_lambda=2.0,
        reg_alpha=0.5,
        subsample=0.75,         
        colsample_bytree=0.7,
        max_delta_step=3,
        random_state=42,
        eval_metric='mlogloss',
        tree_method="hist",
        device="cuda",  
        early_stopping_rounds=40
    )
    
    # dọn VRAM
    print("Đang dọn dẹp RAM & VRAM...")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Đang huấn luyện XGBoost...")
    # train bằng tập train, early stop bằng tập valid
    try:
        xgb_model.fit(
            X_train_emb, y_train,
            sample_weight=final_sample_weights,
            eval_set=[(X_train_emb, y_train), (X_valid_emb, y_valid)],
            verbose=100 
        )
    except Exception as e:
        print(f"Lỗi XGBoost, thử fallback về CPU... Lỗi: {e}")
        xgb_model.set_params(device='cpu')
        xgb_model.fit(
            X_train_emb, y_train,
            sample_weight=final_sample_weights,
            eval_set=[(X_train_emb, y_train), (X_valid_emb, y_valid)],
            verbose=100 
        )
        
    print(f"Huấn luyện xong XGBoost tự động dừng tại vòng tối ưu thứ: {xgb_model.best_iteration}")
    
    # test trên tập test
    print("\n--- ĐÁNH GIÁ TRÊN TẬP TEST MASK % ---")
    y_pred = xgb_model.predict(X_test_emb)
    
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    
    print(f"Accuracy:            {acc*100:.2f}%")
    print(f"F1-Score (Macro):    {f1_macro * 100:.2f}%")
    print(f"F1-Score (Weighted): {f1_weighted * 100:.2f}%\n")
    
    print("Classification Report:")
    print(classification_report(y_test, y_pred, digits=4))
    
    return xgb_model

In [6]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from tqdm.notebook import tqdm
import torch
import gc
from torch_geometric.loader import NeighborLoader

# tương tự như hàm trên nhưng sẽ trả về ma trận nối giữa raw features và embedding GAT để train XGBoost với cả 2 loại đặc trưng
@torch.no_grad()
def extract_concat_features_mask(model, graph_data, mask, device='cpu'):
    model.eval()
    print("Đang khởi tạo Inference Loader lấy Embeddings & Raw Features...")
    
    loader = NeighborLoader(
        graph_data,
        num_neighbors=[10, 5],
        input_nodes=mask,
        batch_size=512,
        shuffle=False,
        num_workers=0 
    )
    
    all_combined = []
    all_labels = []
    
    pbar = tqdm(loader, desc="Đang trích xuất Vectơ Nối")
    for batch in pbar:
        batch = batch.to(device)
        
        _, emb = model(batch.x, batch.edge_index, batch.edge_attr)
        
        bs = batch.batch_size
        
        # lấy raw features và embedding các node gốc
        raw_features = batch.x[:bs].cpu().numpy()
        embeddings = emb[:bs].cpu().numpy()
        
        # concat raw_features và embeddings 
        combined_matrix = np.concatenate([raw_features, embeddings], axis=1)
        
        all_combined.append(combined_matrix)
        all_labels.append(batch.y[:bs].cpu().numpy())
        
        del batch, emb, _, raw_features, embeddings, combined_matrix
        
    pbar.close()
    

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    final_combined = np.concatenate(all_combined, axis=0)
    final_labels = np.concatenate(all_labels, axis=0)
    
    print(f"Trích xuất thành công! Kích thước Ma trận Nối: {final_combined.shape}")
    return final_combined, final_labels

def train_evaluate_concat_xgboost(X_train, y_train, X_valid, y_valid, X_test, y_test):
    print("\n--- BẮT ĐẦU TRAIN XGBOOST TỪ FEATURE + EMBEDDING (CONCAT) ---")

    from sklearn.utils.class_weight import compute_class_weight
    print("Đang tính toán Custom Smoothed Class Weights...")
    
    unique_classes = np.unique(y_train)
    raw_class_weights = compute_class_weight('balanced', classes=unique_classes, y=y_train)
    weights_dict = {c: np.sqrt(w) for c, w in zip(unique_classes, raw_class_weights)}
    
    if 4 in weights_dict: weights_dict[4] *= 0.6
    if 6 in weights_dict: weights_dict[6] *= 1.5
    if 9 in weights_dict: weights_dict[9] *= 2.5
    if 1 in weights_dict: weights_dict[1] *= 0.8

    final_sample_weights = np.array([weights_dict[y] for y in y_train])

    xgb_model = xgb.XGBClassifier(
        n_estimators=1000,
        learning_rate=0.08,
        max_depth=7, 
        min_child_weight=2,
        gamma=0.4,
        reg_lambda=2.0,
        reg_alpha=0.5,
        subsample=0.8,         
        colsample_bytree=0.9,
        max_delta_step=3,
        random_state=42,
        eval_metric='mlogloss',
        tree_method="hist",
        device="cuda",
        early_stopping_rounds=40
    )
    
    print(" Đang dọn dẹp RAM & VRAM (Garbage Collection)...")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Đang huấn luyện XGBoost... (Early Stopping trên Valid Set)")
    try:
        xgb_model.fit(
            X_train, y_train,
            sample_weight=final_sample_weights,
            eval_set=[(X_train, y_train), (X_valid, y_valid)],
            verbose=100 
        )
    except Exception as e:
        print(f"Lỗi XGBoost, thử fallback về CPU... Lỗi: {e}")
        xgb_model.set_params(device='cpu')
        xgb_model.fit(
            X_train, y_train,
            sample_weight=final_sample_weights,
            eval_set=[(X_train, y_train), (X_valid, y_valid)],
            verbose=100 
        )
        
    print(f"Huấn luyện xong XGBoost tự động dừng tại vòng tối ưu thứ: {xgb_model.best_iteration}")
    
    print("\n--- ĐÁNH GIÁ (EVALUATION) TRÊN TẬP TEST MASK ---")
    y_pred = xgb_model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    
    print(f"Accuracy:            {acc*100:.2f}%")
    print(f"F1-Score (Macro):    {f1_macro * 100:.2f}%")
    print(f"F1-Score (Weighted): {f1_weighted * 100:.2f}%\n")
    
    print("Classification Report:")
    print(classification_report(y_test, y_pred, digits=4))
    
    return xgb_model

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Đang Train trên:", device)

print("=== BƯỚC 2: CHUẨN BỊ MÔ HÌNH VÀ HUẤN LUYỆN GAT ===")
import gc
gc.collect()

num_features = super_graph_faiss.x.shape[1] 

# lấy số classs
num_classes = len(torch.unique(super_graph_faiss.y)) 
print(f"Số lượng Classes (Loại tấn công): {num_classes}")

# khởi tạo model GAT
model_faiss = GAT_Embedder(
     in_channels=num_features, 
     hidden_channels=128,      
     embedding_dim=64,        
     num_classes=num_classes, 
     heads=8,
     edge_dropout=0.3,
     edge_dim=4 
).to(device)

# tạo mảng Class_Weights dựa trên tập train
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

train_y_cpu = super_graph_faiss.y[super_graph_faiss.train_mask].cpu().numpy()
unique_classes_arr = np.unique(train_y_cpu)
raw_weights_arr = compute_class_weight('balanced', classes=unique_classes_arr, y=train_y_cpu)

# căn bậc 2 để làm mượt trọng số
smoothed_weights = np.sqrt(raw_weights_arr)
# np.clip: giới hạn giá trị của trọng số trong khoảng [0.1,10.0]
smoothed_weights = np.clip(smoothed_weights, a_min=0.1, a_max=10.0)

# chuyển thành tensor và đưa lên device
gat_class_weights_faiss = torch.tensor(smoothed_weights, dtype=torch.float32).to(device)
print(f"Trọng số cân bằng Class Weights GAT: {gat_class_weights_faiss}")

print("\nBắt đầu quá trình Huấn Luyện GAT!")
# chạy hàm train_gat
model_faiss = train_gat(
    model_faiss, 
    train_loader_faiss, 
    valid_loader_faiss, 
    device, 
    epochs=20, 
    lr=0.0005, 
    class_weights=gat_class_weights_faiss,
    patience=6 
)

import os
os.makedirs("model_final", exist_ok=True)
torch.save(model_faiss.state_dict(), "model_final/gat_embedder.pth")
print("\nThành công! Đã lưu trọng số GAT tĩnh vào thư mục 'model_final/gat_embedder.pth'")

Đang Train trên: cuda
=== BƯỚC 2: CHUẨN BỊ MÔ HÌNH VÀ HUẤN LUYỆN GAT ===
Số lượng Classes (Loại tấn công): 11
Trọng số cân bằng Class Weights GAT: tensor([1.8662, 0.3778, 5.2451, 1.3807, 1.9085, 5.3133, 2.4157, 0.8020, 2.0316,
        1.2901, 1.9295], device='cuda:0')

Bắt đầu quá trình Huấn Luyện GAT!


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 1/20 [Train]:   0%|          | 0/1207 [00:00<?, ?it/s]

Epoch 1/20 [Valid]:   0%|          | 0/279 [00:00<?, ?it/s]

=== Epoch 1 Tổng kết ===
   Train Loss: 1.3196 | Train Macro F1: 0.9277
   Valid Loss: 1.8225   | Valid Macro F1: 0.9379
   Learning Rate hiện tại: 0.000500
Best model. Đã lưu trọng số tốt nhất tại Epoch 1 (Val Macro F1: 0.9379 | Val Loss: 1.8225)


...

Epoch 7/20 [Valid]:   0%|          | 0/279 [00:00<?, ?it/s]

=== Epoch 7 Tổng kết ===
   Train Loss: 1.2533 | Train Macro F1: 0.9807
   Valid Loss: 1.8226   | Valid Macro F1: 0.9352
   Learning Rate hiện tại: 0.000250
Báo động Early Stopping: Không cải thiện F1 5/6 epochs.


Epoch 8/20 [Train]:   0%|          | 0/1207 [00:00<?, ?it/s]

Epoch 8/20 [Valid]:   0%|          | 0/279 [00:00<?, ?it/s]

=== Epoch 8 Tổng kết ===
   Train Loss: 1.2526 | Train Macro F1: 0.9811
   Valid Loss: 1.8136   | Valid Macro F1: 0.9343
   Learning Rate hiện tại: 0.000125
Báo động Early Stopping: Không cải thiện F1 6/6 epochs.

Đã kích hoạt Early Stopping tại Epoch 8! Lùi về checkpoint tốt nhất.

Hoàn tất Train! Đã load lại weights MANG LẠI F1 CAO NHẤT (Val Macro F1: 0.9415) để dùng trích xuất Embedding!

Thành công! Đã lưu trọng số GAT tĩnh vào thư mục 'model_final/gat_embedder.pth'


In [8]:
# giải phóng RAM
import gc
gc.collect()

print("=== BƯỚC 3: TRÍCH XUẤT EMBEDDINGS VÀ HUẤN LUYỆN HYBRID XGBOOST ===")
# trích xuất sang Vector cho XGBoost xử lý
print("\nTrích xuất Vector cho Train Mask")
X_train_faiss, y_train_faiss = extract_embeddings_mask(model_faiss, super_graph_faiss, super_graph_faiss.train_mask, device=device)

print("\nTrích xuất Vector cho Valid Mask")
X_valid_faiss, y_valid_faiss = extract_embeddings_mask(model_faiss, super_graph_faiss, super_graph_faiss.valid_mask, device=device)

print("\nTrích xuất Vector cho Test Mask")
X_test_faiss, y_test_faiss = extract_embeddings_mask(model_faiss, super_graph_faiss, super_graph_faiss.test_mask, device=device)

# huấn luyện và đánh giá XGBoost từ embedding của GAT
hybrid_xgb_faiss = train_evaluate_xgboost(X_train_faiss, y_train_faiss, X_valid_faiss, y_valid_faiss, X_test_faiss, y_test_faiss)

# lưu model xgboost đã huấn luyện
import os
os.makedirs("model_final", exist_ok=True)
hybrid_xgb_faiss.save_model("model_final/GAT_XGB_Hybrid_FAISS_Model.json")
print("\nThành công! Hybrid XGBoost FAISS Model (Base) đã lưu vào 'model_final/GAT_XGB_Hybrid_FAISS_Model.json'")

=== BƯỚC 3: TRÍCH XUẤT EMBEDDINGS VÀ HUẤN LUYỆN HYBRID XGBOOST ===

Trích xuất Vector cho Train Mask
Đang khởi tạo Inference Loader múc lấy Embeddings...


Đang trích xuất Vectơ Embedding:   0%|          | 0/4826 [00:00<?, ?it/s]

Trích xuất thành công! Kích thước Ma trận Embeddings: (2470638, 64)

Trích xuất Vector cho Valid Mask
Đang khởi tạo Inference Loader múc lấy Embeddings...


Đang trích xuất Vectơ Embedding:   0%|          | 0/1114 [00:00<?, ?it/s]

Trích xuất thành công! Kích thước Ma trận Embeddings: (570134, 64)

Trích xuất Vector cho Test Mask
Đang khởi tạo Inference Loader múc lấy Embeddings...


Đang trích xuất Vectơ Embedding:   0%|          | 0/1485 [00:00<?, ?it/s]

Trích xuất thành công! Kích thước Ma trận Embeddings: (760240, 64)

--- BẮT ĐẦU TRAIN XGBOOST TỪ BASE GAT EMBEDDING ---
Đang tính toán Custom Smoothed Class Weights...
Đang dọn dẹp RAM & VRAM...
Đang huấn luyện XGBoost...
[0]	validation_0-mlogloss:1.56087	validation_1-mlogloss:1.56453
[100]	validation_0-mlogloss:0.00049	validation_1-mlogloss:0.04181
[122]	validation_0-mlogloss:0.00013	validation_1-mlogloss:0.04580
Huấn luyện xong XGBoost tự động dừng tại vòng tối ưu thứ: 82

--- ĐÁNH GIÁ TRÊN TẬP TEST MASK % ---


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\core.py:751: UserWarning: [21:47:57] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Accuracy:            97.39%
F1-Score (Macro):    93.19%
F1-Score (Weighted): 97.28%

Classification Report:
              precision    recall  f1-score   support

           0     0.8872    0.9797    0.9312     19846
           1     0.9833    1.0000    0.9916    484077
           2     1.0000    0.9984    0.9992      2515
           3     1.0000    0.9841    0.9920     36253
           4     0.7708    0.8651    0.8153     18979
           5     0.9996    0.9988    0.9992      2451
           6     0.8685    0.7376    0.7977     11847
           7     1.0000    1.0000    1.0000    107436
           8     0.8501    0.9964    0.9174     16746
           9     0.9999    0.6951    0.8201     41523
          10     0.9994    0.9755    0.9874     18567

    accuracy                         0.9739    760240
   macro avg     0.9417    0.9301    0.9319    760240
weighted avg     0.9754    0.9739    0.9728    760240


Thành công! Hybrid XGBoost FAISS Model (Base) đã lưu vào 'model_final/GAT_XGB_

In [9]:
# thử GAT + XGB với CONCAT (Raw Features + Embeddings) để xem có cải thiện không
print("\n=== BƯỚC 4: TRÍCH XUẤT VÀ NỐI (CONCAT) FEATURES + EMBEDDINGS CHO HYBRID XGBOOST ===")
X_train_concat, y_train_concat = extract_concat_features_mask(model_faiss, super_graph_faiss, super_graph_faiss.train_mask, device=device)
X_valid_concat, y_valid_concat = extract_concat_features_mask(model_faiss, super_graph_faiss, super_graph_faiss.valid_mask, device=device)
X_test_concat, y_test_concat = extract_concat_features_mask(model_faiss, super_graph_faiss, super_graph_faiss.test_mask, device=device) 
hybrid_xgb_concat = train_evaluate_concat_xgboost(X_train_concat, y_train_concat, X_valid_concat, y_valid_concat, X_test_concat, y_test_concat)

import os
os.makedirs("model_final", exist_ok=True)
hybrid_xgb_concat.save_model("model_final/GAT_XGB_Hybrid_Concat_Model.json")
print("\nThành công! Hybrid XGBoost Concat Model (Raw + Embeddings) đã lưu vào 'model_final/GAT_XGB_Hybrid_Concat_Model.json'")


=== BƯỚC 4: TRÍCH XUẤT VÀ NỐI (CONCAT) FEATURES + EMBEDDINGS CHO HYBRID XGBOOST ===
Đang khởi tạo Inference Loader lấy Embeddings & Raw Features...


Đang trích xuất Vectơ Nối:   0%|          | 0/4826 [00:00<?, ?it/s]

Trích xuất thành công! Kích thước Ma trận Nối: (2470638, 378)
Đang khởi tạo Inference Loader lấy Embeddings & Raw Features...


Đang trích xuất Vectơ Nối:   0%|          | 0/1114 [00:00<?, ?it/s]

Trích xuất thành công! Kích thước Ma trận Nối: (570134, 378)
Đang khởi tạo Inference Loader lấy Embeddings & Raw Features...


Đang trích xuất Vectơ Nối:   0%|          | 0/1485 [00:00<?, ?it/s]

Trích xuất thành công! Kích thước Ma trận Nối: (760240, 378)

--- BẮT ĐẦU TRAIN XGBOOST TỪ FEATURE + EMBEDDING (CONCAT) ---
Đang tính toán Custom Smoothed Class Weights...
 Đang dọn dẹp RAM & VRAM (Garbage Collection)...
Đang huấn luyện XGBoost... (Early Stopping trên Valid Set)
[0]	validation_0-mlogloss:1.56097	validation_1-mlogloss:1.56432
[100]	validation_0-mlogloss:0.00045	validation_1-mlogloss:0.04315
[122]	validation_0-mlogloss:0.00010	validation_1-mlogloss:0.04617
Huấn luyện xong XGBoost tự động dừng tại vòng tối ưu thứ: 82

--- ĐÁNH GIÁ (EVALUATION) TRÊN TẬP TEST MASK ---
Accuracy:            97.30%
F1-Score (Macro):    93.12%
F1-Score (Weighted): 97.17%

Classification Report:
              precision    recall  f1-score   support

           0     0.8560    0.9969    0.9211     19846
           1     0.9824    1.0000    0.9911    484077
           2     1.0000    1.0000    1.0000      2515
           3     1.0000    0.9713    0.9854     36253
           4     0.7774    0.8631 

In [10]:
# Dùng XGBoost Raw Features làm baseline để so sánh
print("\n=== BƯỚC 5: ĐÁNH GIÁ BASELINE XGBOOST CHỈ RAW FEATURES (KHÔNG DÙNG EMBEDDING) ===")
# Trích xuất Raw Features thuần túy cho từng Mask (Train/Valid/Test)
def extract_raw_features_mask(graph_data, mask):
    print("Đang trích xuất Raw Features thuần túy...")
    features = graph_data.x[mask].cpu().numpy()
    labels = graph_data.y[mask].cpu().numpy()
    print(f"Trích xuất thành công! Kích thước Ma trận Raw Features: {features.shape}")
    return features, labels
X_train_raw, y_train_raw = extract_raw_features_mask(super_graph_faiss, super_graph_faiss.train_mask)
X_valid_raw, y_valid_raw = extract_raw_features_mask(super_graph_faiss, super_graph_faiss.valid_mask)
X_test_raw, y_test_raw = extract_raw_features_mask(super_graph_faiss, super_graph_faiss.test_mask)
baseline_xgb_raw = train_evaluate_xgboost(X_train_raw, y_train_raw, X_valid_raw, y_valid_raw, X_test_raw, y_test_raw)



=== BƯỚC 5: ĐÁNH GIÁ BASELINE XGBOOST CHỈ RAW FEATURES (KHÔNG DÙNG EMBEDDING) ===
Đang trích xuất Raw Features thuần túy...
Trích xuất thành công! Kích thước Ma trận Raw Features: (2470638, 314)
Đang trích xuất Raw Features thuần túy...
Trích xuất thành công! Kích thước Ma trận Raw Features: (570134, 314)
Đang trích xuất Raw Features thuần túy...
Trích xuất thành công! Kích thước Ma trận Raw Features: (760240, 314)

--- BẮT ĐẦU TRAIN XGBOOST TỪ BASE GAT EMBEDDING ---
Đang tính toán Custom Smoothed Class Weights...
Đang dọn dẹp RAM & VRAM...
Đang huấn luyện XGBoost...
[0]	validation_0-mlogloss:1.56874	validation_1-mlogloss:1.56968
[100]	validation_0-mlogloss:0.04775	validation_1-mlogloss:0.13943
[110]	validation_0-mlogloss:0.04594	validation_1-mlogloss:0.14407
Huấn luyện xong XGBoost tự động dừng tại vòng tối ưu thứ: 70

--- ĐÁNH GIÁ TRÊN TẬP TEST MASK % ---
Accuracy:            95.63%
F1-Score (Macro):    85.01%
F1-Score (Weighted): 95.89%

Classification Report:
              precisi

In [11]:
# đánh giá GAT trực tiếp trên tập Test Mask (Không qua XGBoost)
@torch.no_grad()
def evaluate_gat_directly(model, graph_data, mask, device='cpu'):
    model.eval()
    print("Đang khởi tạo NeighborLoader cho đánh giá trực tiếp GAT...")
    loader = NeighborLoader(
        graph_data,
        num_neighbors=[5, 3],
        input_nodes=mask,
        batch_size=16384,
        shuffle=False,
        num_workers=0 
    )
    all_preds = []
    all_labels = []
    pbar = tqdm(loader, desc="Đang đánh giá trực tiếp GAT")
    for batch in pbar:  
        batch = batch.to(device)
        out, _ = model(batch.x, batch.edge_index, batch.edge_attr)
        bs = batch.batch_size
        all_preds.append(out[:bs].cpu().numpy())
        all_labels.append(batch.y[:bs].cpu().numpy())
        del batch, out
    pbar.close()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    final_preds = np.concatenate(all_preds, axis=0)
    final_labels = np.concatenate(all_labels, axis=0)   
    pred_classes = final_preds.argmax(axis=1)
    acc = accuracy_score(final_labels, pred_classes)
    f1_macro = f1_score(final_labels, pred_classes, average='macro')
    f1_weighted = f1_score(final_labels, pred_classes, average='weighted')
    print(f"GAT Đánh giá trực tiếp trên Test Mask:")
    print(f"Accuracy:            {acc*100:.2f}%")
    print(f"F1-Score (Macro):    {f1_macro * 100:.2f}%")
    print(f"F1-Score (Weighted): {f1_weighted * 100:.2f}%\n")
    print("Classification Report:")
    print(classification_report(final_labels, pred_classes, digits=4))
evaluate_gat_directly(model_faiss, super_graph_faiss, super_graph_faiss.test_mask, device=device)

Đang khởi tạo NeighborLoader cho đánh giá trực tiếp GAT...


Đang đánh giá trực tiếp GAT:   0%|          | 0/47 [00:00<?, ?it/s]

GAT Đánh giá trực tiếp trên Test Mask:
Accuracy:            96.26%
F1-Score (Macro):    85.88%
F1-Score (Weighted): 96.37%

Classification Report:
              precision    recall  f1-score   support

           0     0.7633    0.8573    0.8076     19846
           1     1.0000    1.0000    1.0000    484077
           2     0.5652    1.0000    0.7222      2515
           3     1.0000    0.9467    0.9726     36253
           4     0.5905    0.7790    0.6718     18979
           5     0.9859    0.9967    0.9913      2451
           6     0.7774    0.7123    0.7435     11847
           7     1.0000    0.9994    0.9997    107436
           8     0.6612    0.9874    0.7920     16746
           9     0.9999    0.6667    0.8000     41523
          10     0.9996    0.8977    0.9459     18567

    accuracy                         0.9626    760240
   macro avg     0.8494    0.8948    0.8588    760240
weighted avg     0.9712    0.9626    0.9637    760240

